# DKA vs DKA + text at low data (BUSI and full COVID)
Does LLM text still help when the visual side is the **real Dual-Kernel Adapter** (Zhu et al., ICLR 2026) instead of our head-side 5x5 branch?

**Model (follows the paper, Sec. 3 + App. B.2):** frozen ViT-B/16 (ImageNet, the same timm backbone as the pipeline notebooks), a DKA adapter after every one of the 12 transformer blocks:
`x + Up(GELU(DWConv51(Down(x)) + DWConv5(Down(x))))` on the 14x14 patch grid, middle dim 16. Only adapters and heads train; LR 1e-3 for adapters and new projections, 1e-4 for the classifier; 100 epochs; Adam; the **last-epoch** model is evaluated (no test-set model selection).

**Variants** (all on the *same* training subsets and test set as `busi_covid_multimodal_pipeline_fast.ipynb` / `covid_full.ipynb`):

| Variant | Visual | Head | Text |
|---|---|---|---|
| `dka` | ViT + DKA, CLS token | linear (as in the paper) | - |
| `dka_mlp` | ViT + DKA, CLS token | fusion MLP (128-d) | - |
| `dka_text` | ViT + DKA, CLS token | fusion MLP | description + 10 answers (BiomedCLIP) |
| `dka_text_shuffled` | ViT + DKA, CLS token | fusion MLP | another image's text (control) |

`dka_text - dka_mlp` is the clean "does text help on top of DKA" comparison (same head). The final table also pulls in the pipeline's frozen-feature results (linear probe, global MLP, + local, full) for the same subsets.

**v2 (H3 stability):** v1 had 3 collapsed MLP-head runs. v2 adds gradient clipping (1.0), a 5% linear LR warmup, a lower LR (3e-4) for the fusion MLP and projections, a NaN/inf-loss guard, an fp32 option, and optional LP-FT. Results go to `dka_results_v2.csv`; v1 stays in `dka_results.csv`. Every run also saves its per-image test predictions and class probabilities to `dka_preds_v2.csv`.

**Run the pipeline notebooks first** (with the BioVLM-derived questions): this notebook reuses their `split.csv` and caption files and embeds the captions itself.

Differences from the official repo (github.com/misswayguy/DKA), on purpose: its `Adapter.forward` is commented out (the module as published cannot run), it selects the best epoch on the test set, and it uses an implicit-gradient two-optimizer loop the paper does not describe. Here we follow the paper's equation and settings. The paper's zero-init of both Down and Up would freeze the adapter (zero gradients), so only Up is zero-initialised (the adapter starts as an identity).

In [ ]:
# CELL 0 - Setup and config
import os, re, json, time, glob, gc, random, hashlib, math, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def _load_env():                                   # HF_TOKEN etc. from the nearest .env (only needed for downloads)
    d = os.path.abspath(os.getcwd())
    while True:
        f = os.path.join(d, ".env")
        if os.path.isfile(f):
            for line in open(f, encoding="utf8"):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip("'\""))
            return f
        if os.path.dirname(d) == d: return None
        d = os.path.dirname(d)
print(".env:", _load_env())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "./mm_dka"; os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = f"{OUT_DIR}/dka_results_v2.csv"         # v2 = stability fixes below; v1 results stay in dka_results.csv
PREDS_CSV = f"{OUT_DIR}/dka_preds_v2.csv"             # one row per test image per run (for McNemar / H8)

# ---- DKA (paper settings) ----
MID_DIM, K_LARGE, K_SMALL = 16, 51, 5
EPOCHS = 100
BATCH, BIG_BATCH = 8, 32                           # paper uses 8; 32 once a subset has >=128 images (as in the pipeline)
LR_HEAD, LR_MODULES = 1e-4, 1e-3                   # final classifier layer / DKA adapters (paper)
LR_PROJ = 3e-4                                     # H3: fusion MLP + projections (were 1e-3 in v1, where 3 runs collapsed)
HID, P_DROP, P_MOD = 128, 0.3, 0.2                 # fusion head, same as the pipeline
# ---- H3 training stability ----
PRECISION = "fp16"                                 # "fp16" (autocast + loss scaling) or "fp32" (slower, safest); bf16 is slow on RDNA2
AMP = DEVICE == "cuda" and PRECISION == "fp16"
CLIP_NORM = 1.0                                    # gradient-norm clipping (None = off)
WARMUP_FRAC = 0.05                                 # linear LR warmup over the first 5% of steps, then constant
LPFT_EPOCHS = 0                                    # LP-FT: >0 trains only the head for this many epochs before unfreezing adapters

# ---- what to run: dataset -> {fraction: variants}; seeds follow the pipeline ----
ALL_V = ["dka", "dka_mlp", "dka_text", "dka_text_shuffled"]
DATASETS = {
    "BUSI": dict(pipeline_dir="./mm_pipeline_busi", seed=42, pool_frac=0.25,
                 hard_leak=r"benign|malignan|cancer|carcinom|bi-?rads|neoplas",
                 plan={0.0063: ALL_V, 0.0125: ALL_V, 0.125: ALL_V}),
    "COVID_full": dict(pipeline_dir="./mm_pipeline_covid_full", seed=42, pool_frac=0.05,
                       hard_leak=r"covid|sars|coronavirus|pneumoni",
                       plan={0.0063: ALL_V, 0.0125: ["dka", "dka_mlp"]}),   # text at 1.25% only if captioned there
}
SEEDS_FOR = {0.0063: [0, 1, 2, 3, 4], 0.0125: [0, 1, 2, 3, 4], 0.125: [0, 1, 2], 1.0: [0, 1]}
RUN_ORDER = ["BUSI", "COVID_full"]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


In [ ]:
# CELL 1 - Splits and training subsets, rebuilt exactly as in the pipeline notebooks (same split.csv, pool, seeds)
def prepare(name):
    cfg = DATASETS[name]
    df = pd.read_csv(f"{cfg['pipeline_dir']}/split.csv")
    df["id"] = df["image_path"].apply(lambda p: hashlib.md5(p.encode()).hexdigest()[:10] + "_" + os.path.basename(p))
    classes = sorted(df["label"].unique()); y = df["label"].map({c: i for i, c in enumerate(classes)}).values
    tr_all = np.where(df["split"].values == "train")[0]; te = np.where(df["split"].values == "test")[0]
    r = np.random.RandomState(cfg["seed"]); pool = []
    for c in range(len(classes)):
        full = tr_all[y[tr_all] == c]
        pool.append(r.permutation(full)[:max(1, int(round(cfg["pool_frac"] * len(full))))])
    pool = np.concatenate(pool)
    def subset(frac, seed):
        rng = np.random.RandomState(seed); out = []
        for c in range(len(classes)):
            full = tr_all[y[tr_all] == c]; n_c = max(1, int(round(frac * len(full))))
            src = pool[y[pool] == c] if frac <= cfg["pool_frac"] else full
            src = src[rng.permutation(len(src))]; out.append(src[:min(n_c, len(src))])
        return np.concatenate(out)
    # sanity check against the pipeline's own results: same subset sizes for every (fraction, seed)
    _pv2 = f"{cfg['pipeline_dir']}/results_v2.csv"
    pr = pd.read_csv(_pv2 if os.path.exists(_pv2) else f"{cfg['pipeline_dir']}/results_fast.csv")
    for f in cfg["plan"]:
        for s in SEEDS_FOR[f]:
            n = pr[(pr["frac"].round(4) == round(f, 4)) & (pr["seed"] == s)]["n_train"]
            assert n.empty or int(n.iloc[0]) == len(subset(f, s)), f"{name} {f} seed {s}: subset differs from the pipeline"
    return dict(df=df, y=y, classes=classes, test=te, subset=subset, cfg=cfg)

D = {name: prepare(name) for name in RUN_ORDER}
for name, d in D.items():
    print(f"{name}: classes {d['classes']} | test {len(d['test'])} | "
          + ", ".join(f"{f*100:g}% -> {len(d['subset'](f, 0))} imgs" for f in d["cfg"]["plan"]))


In [ ]:
# CELL 2 - Text features: embed the pipeline's current captions with BiomedCLIP (cached per caption file)
import open_clip
def load_text(name):
    d = D[name]; cfg = d["cfg"]
    info = json.load(open(f"{cfg['pipeline_dir']}/run_info.json"))
    caps = os.path.join(cfg["pipeline_dir"], os.path.basename(info["caps_file"]))
    tag = os.path.basename(caps); cache = f"{OUT_DIR}/{name}_text_{tag.replace('.jsonl', '')}.pt"
    N = len(d["df"])
    if os.path.exists(cache):
        t = torch.load(cache); print(name, "text embeddings from cache", tag); return t
    rows = {r["id"]: r for r in map(json.loads, open(caps)) if r.get("parse_ok")}
    leak = re.compile(cfg["hard_leak"], re.I)
    redact = lambda s: re.sub(r"\s+", " ", leak.sub("", s)).strip()
    ids = [i for i in d["df"]["id"] if i in rows]
    model, _ = open_clip.create_model_from_pretrained("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    tok = open_clip.get_tokenizer("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    model = model.to(DEVICE).eval()
    @torch.no_grad()
    def emb(texts, bs=64):
        out = []
        for i in range(0, len(texts), bs):
            out.append(F.normalize(model.encode_text(tok(texts[i:i + bs], context_length=256).to(DEVICE)).float(), dim=-1).cpu())
        return torch.cat(out)
    nq = len(rows[ids[0]]["answers"])
    de = emb([redact(rows[i]["description"]) for i in ids])
    qa = emb([redact(a) for i in ids for a in rows[i]["answers"]]).reshape(len(ids), nq, -1).mean(1)
    pos = {i: k for k, i in enumerate(d["df"]["id"])}
    Dm, Qm, has = torch.zeros(N, 512), torch.zeros(N, 512), torch.zeros(N, dtype=torch.bool)
    for k, i in enumerate(ids):
        Dm[pos[i]] = de[k]; Qm[pos[i]] = F.normalize(qa[k], dim=0); has[pos[i]] = True
    t = dict(D=Dm, Q=Qm, has=has, tag=tag, questions=info.get("questions"))
    torch.save(t, cache); del model; gc.collect(); torch.cuda.empty_cache()
    print(name, f"embedded {len(ids)} captions from {tag}"); return t

TEXT = {name: load_text(name) for name in RUN_ORDER}
for name, t in TEXT.items():
    print(name, "| captions:", t["tag"], "| first question:", (t["questions"] or ["?"])[0][:80])


In [ ]:
# CELL 3 - Load the images the runs need into RAM (uint8, 256x256); crops/flips happen on the GPU
import timm
_vit_cfg = timm.data.resolve_data_config({}, model=timm.create_model("vit_base_patch16_224", pretrained=False))
MEAN = torch.tensor(_vit_cfg["mean"]).view(1, 3, 1, 1); STD = torch.tensor(_vit_cfg["std"]).view(1, 3, 1, 1)

def needed_rows(name):
    d = D[name]; s = set(d["test"].tolist())
    for f in d["cfg"]["plan"]:
        for seed in SEEDS_FOR[f]: s |= set(d["subset"](f, seed).tolist())
    return np.array(sorted(s))

IMG = {}
for name in RUN_ORDER:
    rows = needed_rows(name); df = D[name]["df"]
    arr = torch.empty(len(rows), 3, 256, 256, dtype=torch.uint8)
    for k, r in enumerate(tqdm(rows, desc=f"{name} images")):
        im = Image.open(df["image_path"][r]).convert("RGB").resize((256, 256), Image.BICUBIC)
        arr[k] = torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1)
    row2k = np.full(len(df), -1); row2k[rows] = np.arange(len(rows))
    IMG[name] = dict(x=arr, row2k=row2k)
    print(f"{name}: {len(rows)} images in RAM ({arr.numel() / 1e9:.2f} GB)")

def batch_images(name, rows, train):
    k = torch.as_tensor(IMG[name]["row2k"][rows]); x = IMG[name]["x"][k].to(DEVICE, non_blocking=True).float() / 255
    if train:                                     # random 224 crop of the 256 image + horizontal flip (as in the DKA repo)
        i, j = np.random.randint(0, 33, 2)
        x = x[:, :, i:i + 224, j:j + 224]
        flip = torch.rand(x.size(0), device=x.device) < 0.5
        x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    else:
        x = x[:, :, 16:240, 16:240]               # center crop
    return (x - MEAN.to(x.device)) / STD.to(x.device)


In [ ]:
# CELL 4 - DKA adapter, ViT with DKA in every block, and the heads
class DKA(nn.Module):
    """x + Up(GELU(DWConv_large(Down(x)) + DWConv_small(Down(x)))) on the patch grid; prefix (CLS) tokens skip the convs."""
    def __init__(s, dim=768, mid=MID_DIM, kl=K_LARGE, ks=K_SMALL, n_prefix=1):
        super().__init__()
        s.down = nn.Linear(dim, mid); s.up = nn.Linear(mid, dim); s.n_prefix = n_prefix
        s.large = nn.Conv2d(mid, mid, kl, padding=kl // 2, groups=mid)
        s.small = nn.Conv2d(mid, mid, ks, padding=ks // 2, groups=mid)
        nn.init.zeros_(s.up.weight); nn.init.zeros_(s.up.bias)          # adapter starts as identity
    def forward(s, x):
        B, N, _ = x.shape; h = s.down(x)
        pre, pat = h[:, :s.n_prefix], h[:, s.n_prefix:]
        g = int(round(pat.size(1) ** 0.5))
        p = pat.transpose(1, 2).reshape(B, -1, g, g)
        p = (s.large(p) + s.small(p)).flatten(2).transpose(1, 2)
        return x + s.up(F.gelu(torch.cat([pre, p], 1)))

class BlockDKA(nn.Module):
    def __init__(s, blk, adapter): super().__init__(); s.blk, s.adapter = blk, adapter
    def forward(s, x, *a, **k): return s.adapter(s.blk(x, *a, **k))

class DKAModel(nn.Module):
    def __init__(s, n_cls, head="linear", text=False):
        super().__init__()
        s.vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
        for p in s.vit.parameters(): p.requires_grad = False
        s.vit.blocks = nn.Sequential(*[BlockDKA(b, DKA(n_prefix=s.vit.num_prefix_tokens)) for b in s.vit.blocks])
        s.text, s.kind = text, head
        if head == "linear":
            s.head = nn.Linear(768, n_cls)
        else:                                     # same fusion head as the pipeline's Fusion module
            s.enc = nn.ModuleDict({"g": nn.Sequential(nn.LayerNorm(768), nn.Linear(768, HID))})
            if text:
                s.enc["d"] = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, HID))
                s.enc["q"] = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, HID))
            s.head = nn.Sequential(nn.Dropout(P_DROP), nn.Linear(len(s.enc) * HID, HID), nn.GELU(), nn.Linear(HID, n_cls))
    def forward(s, x, d=None, q=None):
        g = s.vit.forward_features(x)[:, 0]       # CLS after the final norm, as in the pipeline's cached features
        if s.kind == "linear": return s.head(g)
        src = {"g": g, "d": d, "q": q}
        z = torch.stack([s.enc[k](src[k].float()) for k in s.enc], 1)
        if s.training and z.size(1) > 1 and P_MOD > 0:   # modality dropout, never dropping every stream
            m = (torch.rand(z.size(0), z.size(1), device=z.device) > P_MOD).float()
            none = m.sum(1) == 0
            if none.any(): m[none.nonzero().squeeze(1), torch.randint(0, z.size(1), (int(none.sum()),), device=z.device)] = 1
            z = z * m.unsqueeze(-1)
        return s.head(z.flatten(1))

def param_groups(model):
    """final classifier -> LR_HEAD, DKA adapters -> LR_MODULES, fusion projections + MLP hidden layer -> LR_PROJ."""
    final = model.head if model.kind == "linear" else model.head[-1]
    fin = {id(p) for p in final.parameters()}
    ada = [p for n, p in model.named_parameters() if p.requires_grad and ".adapter." in n]
    aid = {id(p) for p in ada}
    rest = [p for p in model.parameters() if p.requires_grad and id(p) not in fin and id(p) not in aid]
    groups = [{"params": list(final.parameters()), "lr": LR_HEAD, "name": "head"},
              {"params": ada, "lr": LR_MODULES, "name": "adapter"}]
    if rest: groups.append({"params": rest, "lr": LR_PROJ, "name": "proj"})
    return groups

_m = DKAModel(3, head="mlp", text=True)
print(f"trainable params: {sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6:.2f} M "
      f"(adapters {sum(p.numel() for n, p in _m.named_parameters() if 'adapter' in n) / 1e6:.2f} M)"); del _m


In [ ]:
# CELL 5 - Training / evaluation engine (resumable; saves metrics and per-image predictions)
from sklearn.metrics import accuracy_score, f1_score, recall_score
SPEC = {"dka": dict(head="linear", text=False), "dka_mlp": dict(head="mlp", text=False),
        "dka_text": dict(head="mlp", text=True), "dka_text_shuffled": dict(head="mlp", text=True, shuffle=True)}
COLS = ["dataset", "variant", "frac", "seed", "n_train", "acc", "f1", "sen", "epochs", "text_tag", "minutes",
        "precision", "clip", "warmup", "lr_proj", "lpft", "skipped_steps"]

def derange(a, rng):
    if len(a) < 2: return a
    while True:
        p = a[rng.permutation(len(a))]
        if (p != a).all(): return p

def train_eval(name, variant, tr, seed):
    d, t, spec = D[name], TEXT[name], SPEC[variant]
    te = d["test"]; set_seed(seed)
    model = DKAModel(len(d["classes"]), spec["head"], spec["text"]).to(DEVICE)
    opt = torch.optim.Adam(param_groups(model)); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    tr_t, te_t = tr.copy(), te.copy()
    if spec.get("shuffle"):
        rng = np.random.RandomState(seed + 999); tr_t, te_t = derange(tr_t, rng), derange(te_t, rng)
    y = torch.as_tensor(d["y"], device=DEVICE).long()
    Dm, Qm = t["D"].to(DEVICE), t["Q"].to(DEVICE)
    n = len(tr); bs = BIG_BATCH if n >= 128 else BATCH
    steps_per_ep = math.ceil(n / bs); total = EPOCHS * steps_per_ep; warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda k: min(1.0, (k + 1) / warm))
    adapters = [p for g in opt.param_groups if g.get("name") == "adapter" for p in g["params"]]
    skipped = 0
    for ep in range(EPOCHS):
        for p in adapters: p.requires_grad_(ep >= LPFT_EPOCHS)          # LP-FT: head first, then adapters
        model.train(); perm = np.random.permutation(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]; rows, trows = tr[b], torch.as_tensor(tr_t[b], device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                out = model(batch_images(name, rows, True), Dm[trows], Qm[trows])
            loss = F.cross_entropy(out.float(), y[torch.as_tensor(rows, device=DEVICE)])
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):                                 # never step on a NaN/inf loss
                skipped += 1; sched.step(); continue
            scaler.scale(loss).backward()
            if CLIP_NORM:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], CLIP_NORM)
            scaler.step(opt); scaler.update(); sched.step()
    model.eval(); logits = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        for i in range(0, len(te), 64):
            rows, trows = te[i:i + 64], torch.as_tensor(te_t[i:i + 64], device=DEVICE)
            logits.append(model(batch_images(name, rows, False), Dm[trows], Qm[trows]).float().cpu())
    prob = torch.cat(logits).softmax(1).numpy(); p = prob.argmax(1); yt = d["y"][te]
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return dict(acc=100 * accuracy_score(yt, p), f1=f1_score(yt, p, average="macro"), sen=100 * recall_score(yt, p, average="macro"),
                skipped_steps=skipped), prob

def load_results():
    r = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=COLS)
    r["text_tag"] = r["text_tag"].fillna(""); return r

def save_preds(name, v, f, s, tag, prob):
    d = D[name]; te = d["test"]
    out = pd.DataFrame({"dataset": name, "variant": v, "frac": round(f, 4), "seed": s, "text_tag": tag,
                        "image_id": d["df"]["id"].values[te], "y_true": d["y"][te], "y_pred": prob.argmax(1)})
    for k in range(prob.shape[1]): out[f"p{k}"] = prob[:, k].round(5)
    out.to_csv(PREDS_CSV, mode="a", header=not os.path.exists(PREDS_CSV), index=False)

def run(name, budget_min=None):
    d, t = D[name], TEXT[name]; t0 = time.time()
    res = load_results(); done = set(zip(res["dataset"], res["variant"], res["frac"].round(4), res["seed"], res["text_tag"]))
    jobs = []
    for f, variants in d["cfg"]["plan"].items():
        for s in SEEDS_FOR[f]:
            tr = d["subset"](f, s)
            for v in variants:
                tag = t["tag"] if SPEC[v]["text"] else ""
                if (name, v, round(f, 4), s, tag) in done: continue
                if SPEC[v]["text"] and not (t["has"][tr].all() and t["has"][d["test"]].all()):
                    print(f"skip {name} {v} {f*100:g}% seed {s}: no captions for these images"); continue
                jobs.append((f, s, v, tr, tag))
    print(f"{name}: {len(jobs)} runs to do")
    for f, s, v, tr, tag in tqdm(jobs, desc=name):
        if budget_min and time.time() - t0 > budget_min * 60:
            print("time budget reached; re-run this cell to continue"); break
        t1 = time.time(); m, prob = train_eval(name, v, tr, s)
        save_preds(name, v, f, s, tag, prob)                              # predictions first, then the results row
        row = dict(dataset=name, variant=v, frac=round(f, 4), seed=s, n_train=len(tr), **m, epochs=EPOCHS,
                   text_tag=tag, minutes=round((time.time() - t1) / 60, 2), precision=PRECISION, clip=CLIP_NORM,
                   warmup=WARMUP_FRAC, lr_proj=LR_PROJ, lpft=LPFT_EPOCHS)
        pd.DataFrame([row])[COLS].to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)
        flag = f"  [{m['skipped_steps']} non-finite steps skipped]" if m["skipped_steps"] else ""
        print(f"  {v:<18} {f*100:>5g}% seed {s}: acc {m['acc']:.1f}  ({row['minutes']:.1f} min){flag}")


In [ ]:
# CELL 6 - BUSI runs (4 / 7 / 78 training images)
run("BUSI")


In [ ]:
# CELL 7 - COVID (full) runs (106 / 211 training images); the slowest part
run("COVID_full")


In [ ]:
# CELL 8 - Results: DKA variants next to the pipeline's frozen-feature variants (same subsets, same test set)
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
res = load_results()
rows = []
for name in RUN_ORDER:
    d, tag = D[name], TEXT[name]["tag"]
    r = res[(res["dataset"] == name) & ((res["text_tag"] == "") | (res["text_tag"] == tag))].copy()
    _pv2 = f"{d['cfg']['pipeline_dir']}/results_v2.csv"
    pr = pd.read_csv(_pv2 if os.path.exists(_pv2) else f"{d['cfg']['pipeline_dir']}/results_fast.csv")
    if "text_tag" in pr: pr = pr[pr["text_tag"].fillna("").isin(["", tag])]
    pr = pr[pr["variant"].isin(["linear_probe", "global_mlp", "global+local", "full"])].assign(dataset=name)
    rows.append(pd.concat([pr[["dataset", "variant", "frac", "seed", "acc", "f1", "sen"]], r[["dataset", "variant", "frac", "seed", "acc", "f1", "sen"]]]))
allr = pd.concat(rows)
order = ["linear_probe", "global_mlp", "global+local", "full", "dka", "dka_mlp", "dka_text", "dka_text_shuffled"]
for name in RUN_ORDER:
    a = allr[allr["dataset"] == name]
    fr = sorted(D[name]["cfg"]["plan"])
    tab = a[a["frac"].round(4).isin([round(f, 4) for f in fr])].groupby(["variant", "frac"])["acc"].agg(["mean", "std", "count"])
    tab = tab.apply(lambda x: f"{x['mean']:.1f} ± {x['std']:.1f} (n={int(x['count'])})", axis=1).unstack("frac").reindex([v for v in order if v in set(a["variant"])])
    tab.columns = [f"{c*100:g}%" for c in tab.columns]
    print(f"\n=== {name}: accuracy % ==="); print(tab.to_string()); tab.to_csv(f"{OUT_DIR}/table_acc_{name}.csv")
    print("\npaired differences (same subsets):")
    for f in fr:
        p = a[a["frac"].round(4) == round(f, 4)].pivot_table(index="seed", columns="variant", values="acc")
        for x, y_ in [("dka_text", "dka_mlp"), ("dka_text", "dka_text_shuffled"), ("dka_mlp", "global_mlp"), ("dka", "linear_probe"), ("dka_text", "full")]:
            if x in p and y_ in p:
                dd = (p[x] - p[y_]).dropna()
                if len(dd) > 1:
                    pv = stats.ttest_rel(p.loc[dd.index, x], p.loc[dd.index, y_]).pvalue
                    print(f"  {f*100:>5g}%  {x} - {y_}: {dd.mean():+.1f} pts  ({(dd > 0).sum()}/{len(dd)} runs better, paired t p={pv:.3f})")
allr.to_csv(f"{OUT_DIR}/all_results_long.csv", index=False)


## Notes for the write-up
- Backbone: timm `vit_base_patch16_224` (ImageNet-21k pretrained, fine-tuned on ImageNet-1k), the same backbone whose frozen features the pipeline notebooks use. The DKA repo uses torchvision's ViT-B/16 (ImageNet-1k).
- Augmentation: random 224 crop of a 256 resize + horizontal flip during training (as in the DKA repo); center crop at test. The pipeline's frozen-feature runs use no augmentation.
- Last-epoch evaluation, no model selection on the test set.
- Mean ± std over the pipeline's independent subset draws (not 5-fold CV).